# lltInstructionData: the lab notebook, recorded run on a Kaggle T4

The page's cells in order. Each cell is wrapped in a timer for the Run Library; the learner notebook has no timers.

In [ ]:
# Runtime check. Run this cell first.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU; Kaggle gives two, and the trainer refuses a DataParallel model
import platform, shutil, sys
ok = True
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    ok = False
    print("Python 3.11 or later is needed, and 3.12 is recommended.")
free = shutil.disk_usage(".").free / 1e9
print("Free disk", round(free, 1), "GB")
if free < 12:
    ok = False
    print("The data shards and five checkpoints need about 8 GB of disk. Free some space.")
import torch
if not torch.cuda.is_available():
    ok = False
    print("No GPU. In Colab use Runtime > Change runtime type > T4 GPU. On Kaggle pick the T4 accelerator, not the P100.")
else:
    name = torch.cuda.get_device_name(0)
    print("GPU", name, "| compute capability", torch.cuda.get_device_capability(0))
    if "P100" in name:
        ok = False
        print("Current PyTorch has no kernels for the P100. Switch the accelerator to T4.")
print("Ready." if ok else "Fix the items above, then run this cell again.")

In [ ]:
# Pinned versions of the recorded runs. torch and numpy stay as the runtime ships them: the constraints file stops pip from changing them.
import numpy, torch
open("constraints.txt", "w").write("numpy==" + numpy.__version__ + chr(10) + "torch==" + torch.__version__ + chr(10))
!pip install -q -c constraints.txt "transformers==5.22.1" "tokenizers==0.23.3" "huggingface_hub==2.4.0" "datasets==5.1.0" "trl==1.17.1" "accelerate==1.15.0" "sentence-transformers==6.2.0" "lm-eval[ifeval]==0.4.13" "langdetect==1.0.9"

## c01_pool.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
import collections, json, os
import numpy as np
import pyarrow.parquet as pq
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Pinned revisions, so every run reads the same rows.
DOLCI = ("allenai/Dolci-Instruct-SFT", "bd3c8f3a9b2cc5a9682e44b96ddd0bb2ff027221", "data/train-00000-of-00015.parquet")
WILD = ("allenai/WildChat-1M", "7d6490e462285cf85d91eabea0f9a954fbddcd1f", "data/train-00000-of-00014.parquet")
N_EACH = 4000

def read_shard(repo, rev, fname, columns):
    path = hf_hub_download(repo, fname, repo_type="dataset", revision=rev)
    return pq.read_table(path, columns=columns)

def first_turn(msgs):
    # The first user message and the assistant reply right after it.
    for a, b in zip(msgs, msgs[1:]):
        if a["role"] == "user" and b["role"] == "assistant" and a["content"] and b["content"]:
            return a["content"], b["content"]
    return None

pool, skipped = [], collections.Counter()
dolci = read_shard(*DOLCI, columns=["id", "messages", "source_dataset"])
for i in np.random.default_rng(0).choice(dolci.num_rows, size=N_EACH, replace=False):
    row = dolci.slice(int(i), 1).to_pylist()[0]
    turn = first_turn(row["messages"])
    if turn is None:
        skipped["dolci"] += 1
        continue
    pool.append({"id": "dolci-" + row["id"], "source": "dolci", "sub": row["source_dataset"],
                 "prompt": turn[0], "response": turn[1]})

wild = read_shard(*WILD, columns=["conversation_hash", "model", "conversation", "language", "redacted"])
for i in np.random.default_rng(1).choice(wild.num_rows, size=N_EACH, replace=False):
    row = wild.slice(int(i), 1).to_pylist()[0]
    turn = first_turn(row["conversation"])
    if turn is None:
        skipped["wildchat"] += 1
        continue
    pool.append({"id": "wild-" + str(i), "source": "wildchat", "sub": row["model"], "lang": row["language"],
                 "redacted": row["redacted"], "prompt": turn[0], "response": turn[1]})

print(dolci.num_rows, "Dolci rows in shard 0 |", wild.num_rows, "WildChat conversations in shard 0")
print(len(pool), "pool rows | skipped, no user and assistant pair", dict(skipped))
print(collections.Counter(r["sub"] for r in pool if r["source"] == "dolci").most_common(8))
print(collections.Counter(r["lang"] for r in pool if r["source"] == "wildchat").most_common(5))

os.makedirs("data", exist_ok=True)
with open("data/pool.jsonl", "w", encoding="utf-8") as f:
    for r in pool:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))

STEP_SECONDS['c01_pool.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c01_pool.py'])


## c02_teacher.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
from transformers import AutoModelForCausalLM

TEACHER = "Qwen/Qwen3-4B-Instruct-2507"   # Apache-2.0, picked by our three-teacher test
N_SYNTH = 300
NL = chr(10)
t_tok = AutoTokenizer.from_pretrained(TEACHER)
t_tok.padding_side = "left"
t_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=t_dtype).to(device).eval()
end_id = t_tok.convert_tokens_to_ids("<|im_end|>")

@torch.no_grad()
def generate(prompts, max_new, sample, batch_size):
    # Returns (text, ended) pairs; ended is False when the reply hit max_new before <|im_end|>.
    outs = []
    for s in range(0, len(prompts), batch_size):
        enc = t_tok(prompts[s:s + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        g = teacher.generate(**enc, max_new_tokens=max_new, do_sample=sample, temperature=1.0 if sample else None,
                             top_p=1.0 if sample else None, top_k=0 if sample else None,
                             eos_token_id=end_id, pad_token_id=t_tok.pad_token_id)
        for row in g[:, enc["input_ids"].shape[1]:]:
            outs.append((t_tok.decode(row, skip_special_tokens=True).strip(), bool((row == end_id).any())))
    return outs

# Magpie: give the teacher only the start of a user turn. It writes a plausible user request itself.
torch.manual_seed(0)
raw = generate(["<|im_start|>user" + NL] * N_SYNTH, max_new=96, sample=True, batch_size=64)
run_on = sum(not ended for _, ended in raw)
# This teacher rarely closes the user turn. It writes the request, a blank line, then starts answering it.
instructions = [text.split(NL + NL)[0].strip() for text, _ in raw]
instructions = [q for q in instructions if len(q) >= 15]

# Then answer each request in the normal chat format, greedy.
chat_prompts = [t_tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
                for q in instructions]
answers = generate(chat_prompts, max_new=512, sample=False, batch_size=32)
synth = [{"id": "synth-" + str(k), "source": "synthetic", "sub": TEACHER, "prompt": q, "response": a, "ended": ended}
         for k, (q, (a, ended)) in enumerate(zip(instructions, answers)) if a]
pool += synth
print(N_SYNTH, "requests sampled |", run_on, "never closed the user turn |", len(synth), "pairs kept | pool now", len(pool))
print("answers that ended within 512 tokens:", sum(r["ended"] for r in synth))
for r in synth[:4]:
    print("-", r["prompt"][:110].replace(NL, " "))
del teacher
if torch.cuda.is_available():
    torch.cuda.empty_cache()

STEP_SECONDS['c02_teacher.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c02_teacher.py'])


## c03_dedup.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
from sentence_transformers import SentenceTransformer

def norm(s):
    return " ".join(s.lower().split())

# Exact duplicates: the same prompt after lower-casing and collapsing whitespace. Keep the first.
first_seen, exact_dup = {}, set()
for i, r in enumerate(pool):
    key = norm(r["prompt"])
    if key in first_seen:
        exact_dup.add(i)
    else:
        first_seen[key] = i

# Near duplicates: cosine similarity of prompt embeddings from a multilingual embedder (released 2026-04-29).
embedder = SentenceTransformer("ibm-granite/granite-embedding-311m-multilingual-r2", device=device,
                               model_kwargs={"dtype": torch.float32})
embedder.max_seq_length = 512                      # the first 512 tokens of each prompt
E = embedder.encode([r["prompt"] for r in pool], batch_size=128, normalize_embeddings=True, convert_to_numpy=True)
keep_idx = [i for i in range(len(pool)) if i not in exact_dup]
Ek = E[keep_idx]
S = Ek @ Ek.T
np.fill_diagonal(S, -1.0)
nearest = S.max(axis=1)

def greedy_dedup(S, threshold):
    # Walk the rows in order; drop a row if it is too similar to a row already kept.
    kept, dropped = [], []
    for j in range(len(S)):
        if kept and S[j, kept].max() >= threshold:
            dropped.append(j)
        else:
            kept.append(j)
    return kept, dropped

NEAR_T = 0.95
print(len(pool), "rows |", len(exact_dup), "exact duplicates")
for t in [0.80, 0.85, 0.90, 0.95, 0.98]:
    kept, dropped = greedy_dedup(S, t)
    print("threshold", t, "| near duplicates dropped", len(dropped))
np.save("data/prompt_embeddings.npy", E)              # reused by the selection step

STEP_SECONDS['c03_dedup.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c03_dedup.py'])


## c04_decontam.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
import json as _json

EVALS = {  # name: (repo, revision, file, text column)
    "IFEval": ("google/IFEval", "966cd89545d6b6acfd7638bc708b98261ca58e84", "ifeval_input_data.jsonl", "prompt"),
    "GSM8K test": ("openai/gsm8k", "740312add88f781978c0658806c59bc2815b9866", "main/test-00000-of-00001.parquet", "question"),
    "HumanEval": ("openai/openai_humaneval", "7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544", "openai_humaneval/test-00000-of-00001.parquet", "prompt"),
    "MMLU test": ("cais/mmlu", "c30699e8356da336a370243923dbaf21066bb9fe", "all/test-00000-of-00001.parquet", "question"),
}

def load_eval(repo, rev, fname, col):
    path = hf_hub_download(repo, fname, repo_type="dataset", revision=rev)
    if fname.endswith(".jsonl"):
        return [_json.loads(line)[col] for line in open(path, encoding="utf-8")]
    return pq.read_table(path, columns=[col]).column(col).to_pylist()

def words(s):
    return "".join(ch if ch.isalnum() else " " for ch in s.lower()).split()

def ngrams(s, n=8):
    w = words(s)
    return {" ".join(w[k:k + n]) for k in range(len(w) - n + 1)}

eval_texts = {name: load_eval(*spec) for name, spec in EVALS.items()}
contaminated = {}
for name, texts in eval_texts.items():
    index = collections.defaultdict(set)          # 8-gram -> eval items that contain it
    sizes = []
    for k, t in enumerate(texts):
        g = ngrams(t)
        sizes.append(len(g))
        for x in g:
            index[x].add(k)
    hits = {}
    for i, r in enumerate(pool):
        counts = collections.Counter(k for x in ngrams(r["prompt"]) for k in index.get(x, ()))
        best = max((c / sizes[k] for k, c in counts.items()), default=0.0)
        if best >= 0.5:                            # at least half of one eval item's 8-grams
            hits[i] = round(best, 3)
    contaminated[name] = hits
    print(name.ljust(11), len(texts), "items |", len(hits), "pool prompts share half or more of one item's 8-grams")

# Paraphrases share few 8-grams, so also compare embeddings of pool prompts and eval prompts.
ifeval_near = []
for name, texts in eval_texts.items():
    Ev = embedder.encode(texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True)
    sim = (E @ Ev.T).max(axis=1)
    if name == "IFEval":
        ifeval_near = np.flatnonzero(sim >= 0.95).tolist()     # IFEval is our eval, so drop close paraphrases too
    for t in [0.9, 0.95]:
        near = set(np.flatnonzero(sim >= t).tolist())
        print(name.ljust(11), "cosine", t, "or more to an eval prompt:", len(near), "| not caught by 8-grams:",
              len(near - set(contaminated[name])))

STEP_SECONDS['c04_decontam.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c04_decontam.py'])


## c05_pii.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
import re
from transformers import pipeline

# Patterns written without backslashes: [0-9] for a digit, [.] for a literal dot.
PATTERNS = {
    "email": re.compile("[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+([.][A-Za-z0-9-]+)*[.][A-Za-z]{2,}"),
    "phone": re.compile("(?<![0-9])[+]?[0-9]{1,3}?[ .(-]{0,2}[0-9]{3}[ .)-]{0,2}[0-9]{3}[ .-]?[0-9]{4}(?![0-9])"),
    "ipv4": re.compile("(?<![0-9.])(?:[0-9]{1,3}[.]){3}[0-9]{1,3}(?![0-9.])"),
    "card": re.compile("(?<![0-9])(?:[0-9][ -]?){13,16}(?![0-9])"),
}

def luhn_ok(digits):
    total = 0
    for k, d in enumerate(reversed(digits)):
        d = int(d) * (2 if k % 2 else 1)
        total += d - 9 if d > 9 else d
    return total % 10 == 0

def regex_hits(text):
    found = {}
    for kind, pat in PATTERNS.items():
        ms = [m.group(0) for m in pat.finditer(text)]
        if kind == "card":
            ms = [m for m in ms if luhn_ok([c for c in m if c.isdigit()])]
        if ms:
            found[kind] = len(ms)
    return found

user_rows = [i for i, r in enumerate(pool) if r["source"] == "wildchat"]
pii = {i: regex_hits(pool[i]["prompt"]) for i in user_rows}
pii = {i: h for i, h in pii.items() if h}
by_kind = collections.Counter(k for h in pii.values() for k in h)
print(len(user_rows), "user prompts from WildChat |", len(pii), "with a regex hit |", dict(by_kind))
print("WildChat's own redacted flag on these rows:", sum(bool(pool[i]["redacted"]) for i in user_rows))

# Names: a named-entity recognition (NER) model tags person names in English prompts.
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)
english = [i for i in user_rows if pool[i]["lang"] == "English"]
names = {}
for i, ents in zip(english, ner([pool[i]["prompt"][:1500] for i in english], batch_size=32)):
    people = [e["word"] for e in ents if e["entity_group"] == "PER" and e["score"] >= 0.9]
    if people:
        names[i] = people
print(len(english), "English user prompts |", len(names), "name at least one person (NER score 0.9 or more)")

STEP_SECONDS['c05_pii.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c05_pii.py'])


## c06_reward.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
import gzip
from transformers import AutoModelForSequenceClassification

RM = "Skywork/Skywork-Reward-V2-Qwen3-0.6B"
rm_tok = AutoTokenizer.from_pretrained(RM)
rm_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
rm = AutoModelForSequenceClassification.from_pretrained(RM, dtype=rm_dtype, num_labels=1).to(device).eval()

@torch.no_grad()
def reward(pairs, batch_size=8, max_len=2048):
    # One scalar per (prompt, response). Shortest first, so each batch pads little.
    ids = [rm_tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                      tokenize=True)["input_ids"][:max_len] for p, a in pairs]
    order = np.argsort([len(x) for x in ids])
    out = np.zeros(len(ids), dtype=np.float32)
    for s in range(0, len(ids), batch_size):
        idx = order[s:s + batch_size]
        batch = rm_tok.pad({"input_ids": [ids[k] for k in idx]}, return_tensors="pt").to(device)
        out[idx] = rm(**batch).logits[:, 0].float().cpu().numpy()
    return out

# Calibrate on human labels first: HelpSteer2 validation, single-turn rows, helpfulness rated 0 to 4.
path = hf_hub_download("nvidia/HelpSteer2", "validation.jsonl.gz", repo_type="dataset",
                       revision="990b2711a36180dd19d9c94b8627844866f8982a")
hs = [json.loads(line) for line in gzip.open(path, "rt", encoding="utf-8")]
hs = [h for h in hs if "<extra_id_1>" not in h["prompt"]]
hs_score = reward([(h["prompt"], h["response"]) for h in hs])
helpful = np.array([h["helpfulness"] for h in hs])
by_prompt = collections.defaultdict(list)
for k, h in enumerate(hs):
    by_prompt[h["prompt"]].append(k)
pairs = [(a, b) for ks in by_prompt.values() for a in ks for b in ks if a < b and helpful[a] != helpful[b]]
agree = np.mean([(hs_score[a] > hs_score[b]) == (helpful[a] > helpful[b]) for a, b in pairs])
print(len(hs), "single-turn HelpSteer2 responses |", len(pairs), "pairs with different human helpfulness")
print("reward model agrees with the human order on", round(float(agree), 3), "of pairs")
for q in [0.25, 0.5, 0.75]:
    t = float(np.quantile(hs_score, q))
    print("keep score >=", round(t, 2), "| kept", int((hs_score >= t).sum()),
          "| share rated 3 or 4:", round(float((helpful[hs_score >= t] >= 3).mean()), 3),
          "| dropped share rated 3 or 4:", round(float((helpful[hs_score < t] >= 3).mean()), 3))

RM_FLOOR = float(np.quantile(hs_score, 0.5))
scores = reward([(r["prompt"], r["response"]) for r in pool])
for src in ["dolci", "wildchat", "synthetic"]:
    m = np.array([r["source"] == src for r in pool])
    if m.any():
        print(src.ljust(9), "rows", int(m.sum()), "| median reward", round(float(np.median(scores[m])), 2))

STEP_SECONDS['c06_reward.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c06_reward.py'])


## c07_select.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
from langdetect import DetectorFactory, detect
from sklearn.cluster import KMeans

DetectorFactory.seed = 0
def is_english(r):
    if r["source"] == "wildchat":
        return r["lang"] == "English"
    try:
        return detect(r["prompt"][:1000]) == "en"
    except Exception:
        return False

def vendi(X):
    # Vendi score: exp of the entropy of the eigenvalues of the similarity matrix divided by n.
    # It reads as the effective number of distinct items, from 1 (all the same) to n (all orthogonal).
    lam = np.linalg.eigvalsh(X @ X.T / len(X))
    lam = lam[lam > 1e-12]
    return float(np.exp(-(lam * np.log(lam)).sum()))

def mean_cos(X):
    n = len(X)
    return float(((X @ X.T).sum() - n) / (n * (n - 1)))

kept_near, _ = greedy_dedup(S, NEAR_T)
unique_rows = {keep_idx[j] for j in kept_near}
contam = set().union(*[set(h) for h in contaminated.values()]) | set(ifeval_near)
english = {i for i, r in enumerate(pool) if is_english(r)}
floor = float(RM_FLOOR)
stages = [("pool", set(range(len(pool)))), ("no exact or near duplicate", unique_rows), ("not contaminated", unique_rows - contam),
          ("no personal data hit", unique_rows - contam - set(pii)), ("English", (unique_rows - contam - set(pii)) & english)]
stages.append(("answer not cut off", {i for i in stages[-1][1] if pool[i].get("ended", True)}))
stages.append(("reward score " + str(round(floor, 2)) + " or more", {i for i in stages[-1][1] if scores[i] >= floor}))
for label, rows in stages:
    print(label.ljust(28), len(rows), dict(collections.Counter(pool[i]["source"] for i in rows)))

# One cluster per topic region of prompt space; take the best-scored rows from each cluster in turn.
cand = sorted(stages[-1][1])
km = KMeans(n_clusters=min(100, len(cand)), random_state=0, n_init=3).fit(E[cand])
per = collections.defaultdict(list)
for i, c in zip(cand, km.labels_):
    per[c].append(i)
queues = [sorted(v, key=lambda i: -scores[i]) for v in per.values()]
curated, target = [], min(1000, len(cand))
while len(curated) < target:
    for q in queues:
        if q and len(curated) < target:
            curated.append(q.pop(0))

eligible = sorted(set(range(len(pool))) - contam)          # bulk arms skip only decontamination
arms = {"curated-1k": curated}
for seed in [0, 1]:
    pick = np.random.default_rng(seed).permutation(eligible)
    arms["bulk-1k-s" + str(seed)], arms["bulk-4k-s" + str(seed)] = list(pick[:1000]), list(pick[:4000])
for name, rows in arms.items():
    X = E[rows[:1000]]
    print(name.ljust(12), "| mean cosine", round(mean_cos(X), 3), "| Vendi score", round(vendi(X), 1),
          "| median reward", round(float(np.median(scores[rows])), 2), "|", dict(collections.Counter(pool[i]["source"] for i in rows)))

with open("data/arms.json", "w", encoding="utf-8") as f:
    json.dump({k: [int(i) for i in v] for k, v in arms.items()}, f)
with open("data/pool_scored.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(pool):
        f.write(json.dumps(dict(r, reward=float(scores[i])), ensure_ascii=False) + chr(10))
flags = {"exact_dup": sorted(exact_dup), "near_kept_rows": sorted(unique_rows), "contaminated": {k: sorted(v) for k, v in contaminated.items()},
         "ifeval_near": ifeval_near, "pii": {str(i): h for i, h in pii.items()}, "names": {str(i): v for i, v in names.items()},
         "english": sorted(english), "rm_floor": floor, "near_threshold": NEAR_T}
with open("data/flags.json", "w", encoding="utf-8") as f:
    json.dump(flags, f, ensure_ascii=False, default=int)

STEP_SECONDS['c07_select.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c07_select.py'])


## c08_train.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

BASE = "HuggingFaceTB/SmolLM2-360M"
STEPS = int(os.environ.get("LLT_STEPS", "250"))   # the same 250 steps of 16 examples for every arm
TRAIN_LOGS = {}

def sft_tokenizer():
    # The base model ships no chat template. Borrow the instruct model's, and stop on its end-of-turn token.
    tok = AutoTokenizer.from_pretrained(BASE)
    inst = AutoTokenizer.from_pretrained(BASE + "-Instruct")
    tok.chat_template, tok.eos_token = inst.chat_template, inst.eos_token
    tok.pad_token = tok.unk_token
    return tok

def train_arm(rows, name, seed):
    data = Dataset.from_list([{"prompt": [{"role": "user", "content": r["prompt"]}],
                               "completion": [{"role": "assistant", "content": r["response"]}]} for r in rows])
    tok = sft_tokenizer()
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
    args = SFTConfig(output_dir="runs/" + name, seed=seed, max_steps=STEPS, per_device_train_batch_size=8,
                     gradient_accumulation_steps=2, learning_rate=5e-5, warmup_steps=10, max_length=1024,
                     completion_only_loss=True, gradient_checkpointing=True, fp16=torch.cuda.is_available(), bf16=False,
                     logging_steps=10, save_strategy="no", report_to="none")
    trainer = SFTTrainer(model=model, args=args, train_dataset=data, processing_class=tok)
    out = trainer.train()
    losses = [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h]
    TRAIN_LOGS[name + "-s" + str(seed)] = {"loss": losses, "seconds": round(out.metrics["train_runtime"], 1)}
    print(name, "seed", seed, "| examples", len(rows), "| epochs", round(STEPS * 16 / len(rows), 2), "| loss", losses[0], "->", losses[-1])
    return trainer.model.eval(), tok

STEP_SECONDS['c08_train.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c08_train.py'])


## c09_eval.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
from lm_eval.tasks.ifeval.utils import process_results

ifeval_path = hf_hub_download("google/IFEval", "ifeval_input_data.jsonl", repo_type="dataset",
                              revision="966cd89545d6b6acfd7638bc708b98261ca58e84")
ifeval = [json.loads(line) for line in open(ifeval_path, encoding="utf-8")]
N_IFEVAL = int(os.environ.get("LLT_N_IFEVAL", str(len(ifeval))))

@torch.no_grad()
def answer_all(model, tok, prompts, max_new=512, batch_size=64):
    # Greedy answers in the chat format, left-padded, stopped at the end-of-turn token.
    tok.padding_side = "left"
    model.to(device)
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
             for p in prompts]
    outs = []
    for s in range(0, len(texts), batch_size):
        enc = tok(texts[s:s + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, eos_token_id=tok.eos_token_id,
                           pad_token_id=tok.pad_token_id)
        new = g[:, enc["input_ids"].shape[1]:]
        outs += [(tok.decode(row, skip_special_tokens=True), bool((row == tok.eos_token_id).any())) for row in new]
    return outs

def ifeval_scores(model, tok):
    docs = ifeval[:N_IFEVAL]
    outs = answer_all(model, tok, [d["prompt"] for d in docs])
    res = [process_results(d, [text]) for d, (text, _) in zip(docs, outs)]
    loose = np.array([r["prompt_level_loose_acc"] for r in res], dtype=float)
    strict = np.array([r["prompt_level_strict_acc"] for r in res], dtype=float)
    stopped = np.mean([s for _, s in outs])
    return loose, strict, stopped, [t for t, _ in outs]

STEP_SECONDS['c09_eval.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c09_eval.py'])


## c10_experiment.py

In [ ]:

import time as _t
STEP_SECONDS = globals().get("STEP_SECONDS", {})
_T0 = _t.time()
def paired_bootstrap(a, b, n_boot=10000, seed=0):
    # Mean of b - a over the same IFEval prompts, with a 95% interval from resampling prompts.
    d = np.asarray(b, dtype=float) - np.asarray(a, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(d), size=(n_boot, len(d)))
    boots = d[idx].mean(axis=1)
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

results = {}
for name in ["curated-1k", "bulk-1k-s0"]:
    model, tok = train_arm([pool[i] for i in arms[name]], name, seed=0)
    loose, strict, stopped, texts = ifeval_scores(model, tok)
    results[name] = loose
    print(name.ljust(11), "| IFEval prompt-level loose", round(loose.mean(), 3), "| strict", round(strict.mean(), 3),
          "| answers that ended with the end token", round(stopped, 3))
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

diff, lo, hi = paired_bootstrap(results["bulk-1k-s0"], results["curated-1k"])
print("curated minus bulk, IFEval loose:", round(diff, 3), "| 95% interval", round(lo, 3), "to", round(hi, 3))

STEP_SECONDS['c10_experiment.py'] = round(_t.time() - _T0, 1)
print('cell seconds', STEP_SECONDS['c10_experiment.py'])


In [ ]:

import json, platform, time
import transformers, trl, datasets, sentence_transformers
rec = {"date": time.strftime("%Y-%m-%d %H:%M"), "gpu": torch.cuda.get_device_name(0),
       "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
               "transformers": transformers.__version__, "trl": trl.__version__, "datasets": datasets.__version__,
               "sentence_transformers": sentence_transformers.__version__, "numpy": np.__version__},
       "trainLogs": TRAIN_LOGS, "ifevalLoose": {k: v.tolist() for k, v in results.items()},
       "stepSeconds": STEP_SECONDS}
json.dump(rec, open("insdata_notebook_record.json", "w"), indent=1)
print(json.dumps({k: v for k, v in rec.items() if k != "ifevalLoose"}, indent=1)[:3000])
